# T5-small Fine-Tuning on Spider

Runs on Colab Free (T4). Checkpoints saved to Google Drive so session disconnects don't kill progress.

**Before running:** upload `train_spider.json`, `train_others.json`, `dev.json`, `tables.json` to `/MyDrive/text-to-sql/spider_data/` in your Google Drive.

## 1. Setup

In [ ]:
!pip install -q transformers datasets accelerate sqlglot

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/text-to-sql')
SPIDER_DIR = DRIVE_ROOT / 'spider_data'
CHECKPOINT_DIR = DRIVE_ROOT / 't5_small_spider'
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

assert (SPIDER_DIR / 'train_spider.json').exists(), 'Upload Spider JSONs to Drive first'
print('Spider dir ok')

In [ ]:
import torch
import random
import numpy as np

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name()}')

## 2. Load Spider

Same preprocessing philosophy as the LSTM notebook: serialize the schema into the input so the model has access to table and column names.

In [ ]:
import json

def load_json(path):
    with open(path) as f:
        return json.load(f)

train = load_json(SPIDER_DIR / 'train_spider.json')
train_others = load_json(SPIDER_DIR / 'train_others.json')
validation = load_json(SPIDER_DIR / 'dev.json')
tables = load_json(SPIDER_DIR / 'tables.json')

print(f'train: {len(train)}  train_others: {len(train_others)}  validation: {len(validation)}')

In [ ]:
def build_schema_string(db_table_info):
    parts = []
    table_names = db_table_info['table_names_original']
    columns = db_table_info['column_names_original']

    tables_to_cols = {i: [] for i in range(len(table_names))}
    for tbl_idx, col_name in columns:
        if tbl_idx == -1:
            continue
        tables_to_cols[tbl_idx].append(col_name)

    for i, tbl in enumerate(table_names):
        cols = ', '.join(tables_to_cols[i])
        parts.append(f'{tbl}({cols})')

    return ' | '.join(parts)

schema_by_db = {t['db_id']: build_schema_string(t) for t in tables}

sample_db = next(iter(schema_by_db))
print(f'{sample_db}: {schema_by_db[sample_db][:200]}...')

In [ ]:
def make_input(example, schema_by_db):
    schema = schema_by_db[example['db_id']]
    return f"translate to SQL: {example['question']} | schema: {schema}"

def to_t5_pairs(examples):
    return [
        {'input': make_input(ex, schema_by_db), 'target': ex['query']}
        for ex in examples
    ]

train_pairs = to_t5_pairs(train) + to_t5_pairs(train_others)
val_pairs = to_t5_pairs(validation)

print(f'train pairs: {len(train_pairs)}  val pairs: {len(val_pairs)}')
print()
print('example input:')
print(train_pairs[0]['input'][:300])
print()
print('example target:')
print(train_pairs[0]['target'])

## 3. Tokenize

In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = 't5-small'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

MAX_INPUT_LEN = 512
MAX_TARGET_LEN = 256

In [ ]:
from datasets import Dataset

def tokenize_fn(batch):
    model_inputs = tokenizer(
        batch['input'],
        max_length=MAX_INPUT_LEN,
        truncation=True,
    )
    labels = tokenizer(
        text_target=batch['target'],
        max_length=MAX_TARGET_LEN,
        truncation=True,
    )
    model_inputs['labels'] = labels['input_ids']
    return model_inputs

train_ds = Dataset.from_list(train_pairs).map(
    tokenize_fn, batched=True, remove_columns=['input', 'target']
)
val_ds = Dataset.from_list(val_pairs).map(
    tokenize_fn, batched=True, remove_columns=['input', 'target']
)

print(train_ds)
print(val_ds)

## 4. Fine-Tune

In [ ]:
from transformers import (
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)

model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
collator = DataCollatorForSeq2Seq(tokenizer, model=model)

In [ ]:
args = Seq2SeqTrainingArguments(
    output_dir=str(CHECKPOINT_DIR),
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=1,
    learning_rate=3e-4,
    weight_decay=0.01,
    warmup_ratio=0.1,
    lr_scheduler_type='linear',
    logging_steps=100,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=2,
    predict_with_generate=False,
    fp16=True,
    report_to='none',
    seed=SEED,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    data_collator=collator,
)

In [ ]:
trainer.train()

In [ ]:
FINAL_DIR = CHECKPOINT_DIR / 'final'
trainer.save_model(str(FINAL_DIR))
tokenizer.save_pretrained(str(FINAL_DIR))
print(f'Saved to {FINAL_DIR}')

## 5. Quick sanity check

In [ ]:
model.eval()
for ex in val_pairs[:3]:
    inputs = tokenizer(ex['input'], return_tensors='pt', truncation=True, max_length=MAX_INPUT_LEN).to(device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=256, num_beams=4)
    pred = tokenizer.decode(out[0], skip_special_tokens=True)
    print('GOLD:', ex['target'])
    print('PRED:', pred)
    print()